In [ ]:
#importing relevant libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.decomposition import TruncatedSVD
import os


#Load and Inspect Data
df = pd.read_excel('Customer_Interactions.xlsx')

In [ ]:
# Show rows
df

In [ ]:
df.shape

In [ ]:
#let do some Exploratory Data Analysis (EDA)
ctr = df["clicked"].mean()
purchase_rate = df["purchased"].mean()

print("CTR:", round(ctr,4), "Purchase rate:", round(purchase_rate,4))

In [ ]:
df.describe()

In [ ]:
# Plot CTR(click rate) over time (7-day rolling)
ctr_by_day = df.set_index("timestamp").resample("7D")["clicked"].mean()
plt.figure(figsize=(7,2.5))
plt.plot(ctr_by_day.index, ctr_by_day.values)
plt.title("7-day rolling CTR (synthetic)")
plt.xlabel("Date")
plt.ylabel("CTR")
plt.tight_layout()
plt.show()

In [ ]:
new_dataset = df.copy()

In [ ]:
#show new_dataset
new_dataset

In [ ]:
# Prepare advetisement dataset and simple features
new_dataset = df.copy()
new_dataset["hour"] = new_dataset["timestamp"].dt.hour
new_dataset["dayofweek"] = new_dataset["timestamp"].dt.dayofweek

#selecting features(columns for training model)
new_df = new_dataset[["age","gender","user_id","product_id","region","device","category","price","rating","hour","dayofweek","clicked","purchased"]].copy()



In [ ]:
#Show new data
new_df

In [ ]:
#converting categorical features to numerical using the one-hot encoding function

cat_c = ["gender","user_id","product_id","region","device","category","hour","dayofweek"]
X = pd.get_dummies(new_df.drop(columns=["clicked"]), columns=cat_c, drop_first=True)
y = new_df["clicked"]

In [ ]:
# Train/test split (smaller)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
#Define Random Forest and Hyperparameter Grid

rfc = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

In [ ]:
# RandomForest Model
rfc = RandomForestClassifier(n_estimators=80, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1)
rfc.fit(X_train, y_train)
y_pred = rfc.predict(X_test)
y_proba = rfc.predict_proba(X_test)[:,1]

print("\nAd-click classifier report:")
print(classification_report(y_test, y_pred, digits=3))
try:
    auc = roc_auc_score(y_test, y_proba)
    print("ROC AUC:", round(auc,4))
except Exception as e:
    print("ROC AUC not available:", e)

In [ ]:
# Building a user-item matrix with implicit weighting and run TruncatedSVD (smaller components)

new_df["weight"] = 0.05 + new_df["clicked"]*0.5 + new_df["purchased"]*2.5
user_item = new_df.groupby(["user_id","product_id"])["weight"].sum().unstack(fill_value=0)
n_components = 20
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item)
item_factors = svd.components_.T
user_index = list(user_item.index)
item_index = list(user_item.columns)

def recommend_for_user(user_id, n=6):
    if user_id not in user_index:
        return []
    ui = user_index.index(user_id)
    uvec = user_factors[ui]
    scores = item_factors.dot(uvec)
    seen = set(user_item.loc[user_id][user_item.loc[user_id] > 0].index)
    ranked = sorted(zip(item_index, scores), key=lambda x: x[1], reverse=True)
    recs = [pid for pid, sc in ranked if pid not in seen][:n]
    return recs

sample_user = new_df["user_id"].sample(1, random_state=42).iloc[0]
recs = recommend_for_user(sample_user, n=8)

print("\nSample recommendations for", sample_user, "->", recs)

In [ ]:
# Integration: rank those recommendations by predicted click probability
def rank_recs_by_click(user_id, recs):
    rows = []
    for pid in recs:
        prod = new_df[new_df["product_id"]==pid].iloc[0]
        usr = new_df[new_df["user_id"]==user_id].iloc[0]
        row = {"age": usr["age"], "price": prod["price"], "rating": prod["rating"],
               "gender": usr["gender"], "region": usr["region"], "device": usr["device"],
               "category": prod["category"], "hour": 12, "dayofweek": 2}
        rows.append((pid, row))
    df = pd.DataFrame([r for _, r in rows])
    df_enc = pd.get_dummies(df, columns=["gender","region","device","category","hour","dayofweek"], drop_first=True)
    # align columns with training data
    for col in X_train.columns:
        if col not in df_enc.columns:
            df_enc[col] = 0
    df_enc = df_enc[X_train.columns]
    probs = rfc.predict_proba(df_enc)[:,1]
    return sorted(zip([r[0] for r in rows], probs), key=lambda x: x[1], reverse=True)

ranked = rank_recs_by_click(sample_user, recs)
print("\nRanked recommended products by predicted click probability:")
for pid, p in ranked:
    print(pid, round(p,3))

In [ ]:
# Create a local folder inside your current working directory
os.makedirs("ecom_synthetic", exist_ok=True)

# Save CSVs and models locally
new_df.to_csv("ecom_synthetic/customers.csv", index=False)
#products.to_csv("ecom_synthetic/products.csv", index=False)
#interactions.to_csv("ecom_synthetic/interactions.csv", index=False)
joblib.dump(rfc, "ecom_synthetic/ad_click_model_rf.joblib")
joblib.dump(svd, "ecom_synthetic/recommender_svd.joblib")

In [ ]:
%run ecom.py